In [ ]:
# ==============================================================================
# 🏆 AN2DL CHALLENGE 2 - RESNET50 TRAINING & INFERENCE PIPELINE (v1.0.4)
# ==============================================================================
# PIPELINE OVERVIEW:
# 1. DATA ACQUISITION: Downloads specific cropped dataset (224px, overlap 56).
# 2. TRAINING: Fine-tunes ResNet50 with custom head, MixUp augmentation, 
#    and Patient-Aware Stratified Splitting.
# 3. INFERENCE: Performs mask-guided tiling on raw test images with Top-K voting.
# ==============================================================================

import os
import cv2
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import shutil
import warnings
import subprocess
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import Dataset, DataLoader

# --- 1. ENVIRONMENT SETUP ---
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚙️  Hardware: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

def set_seed(seed=42):
    """Sets the seed for reproducibility across all libraries."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True 

set_seed(42)

# --- 2. DATA DOWNLOAD UTILITIES ---
try:
    import gdown
except ImportError:
    subprocess.check_call(["pip", "install", "-q", "gdown"])
    import gdown

# Google Drive File IDs for the required datasets
FILES = {
    # Pre-processed training crops (224x224)
    "crops_data.zip": "1Kit41dsZPNYHNJHml6Wp7JK1yHom1VwK", 
    
    # Label CSV for the crops
    "crops_labels.csv": "17XPlHzQsI4_CUFJletOlVvWRdMuAFkzS",
    
    # Raw test data for inference
    "test_data.zip":  "1UTRi8b5z_MQSis0IS6L91BqFisTVRgHj"
}

WORK_DIR = os.getcwd()
INPUT_DIR = os.path.join(WORK_DIR, "grumpy-data")

print("\n⬇️  Downloading Dataset Files...")
if not os.path.exists(INPUT_DIR): os.makedirs(INPUT_DIR)

for name, fid in FILES.items():
    path = os.path.join(INPUT_DIR, name)
    if "PASTE" in fid:
        print(f"⚠️  SKIPPING {name} - ID missing in configuration.")
        continue
    if not os.path.exists(path):
        gdown.download(f'https://drive.google.com/uc?id={fid}', path, quiet=False)

# Extract Archives
print("📦  Extracting Data...")
for zfile in ["crops_data.zip", "test_data.zip"]:
    zpath = os.path.join(INPUT_DIR, zfile)
    if os.path.exists(zpath):
        with zipfile.ZipFile(zpath, 'r') as z:
            z.extractall(INPUT_DIR)

# --- 3. PATH CONFIGURATION ---
def find_folder(base, marker_file_ext=".png"):
    """Recursively finds the folder containing specific file types."""
    for root, _, files in os.walk(base):
        if any(f.endswith(marker_file_ext) for f in files): return root
    return base

# Directory definitions
TRAIN_CROPS_DIR = find_folder(os.path.join(INPUT_DIR, "train_data_crops_crop224_ov56"))
TEST_RAW_DIR = find_folder(os.path.join(INPUT_DIR, "test_data"))
TRAIN_CSV_PATH = os.path.join(INPUT_DIR, "crops_labels.csv")

print(f"\n📂  Training Directory: {TRAIN_CROPS_DIR}")
print(f"📂  Testing Directory:  {TEST_RAW_DIR}")

# --- 4. DATASET IMPLEMENTATION ---
class GrumpyDataset(Dataset):
    """
    Custom Dataset class handling both training (pre-cropped patches)
    and testing (raw full-size images) scenarios.
    """
    def __init__(self, mode, df, data_dir, transform=None):
        self.mode = mode
        self.df = df
        self.data_dir = data_dir
        self.transform = transform
        self.map = {'Luminal A': 0, 'Luminal B': 1, 'HER2(+)': 2, 'Triple negative': 3}
        # ImageNet normalization statistics
        self.norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        if self.mode == 'train':
            # Training Mode: Load specific crop based on dataframe index
            fname = self.df.iloc[idx]['sample_index']
            label_str = self.df.iloc[idx]['label']
            target = torch.tensor(self.map[label_str], dtype=torch.long)
        else:
            # Test Mode: Load file name only (dummy target)
            fname = self.df.iloc[idx]['sample_index']
            target = torch.tensor(-1, dtype=torch.long)
            
        path = os.path.join(self.data_dir, fname)
        img = cv2.imread(path)
        
        # Error handling for missing or corrupt files
        if img is None: 
            img = np.zeros((224, 224, 3), dtype=np.uint8)
            
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = (img/255.0).astype(np.float32)
        
        t = torch.from_numpy(img).permute(2,0,1)
        
        if self.transform: t = self.transform(t)
        t = self.norm(t)
        
        return t, target

    def get_raw_test_data(self, idx):
        """Helper to load full-size image and corresponding mask for inference tiling."""
        fname = self.df.iloc[idx]['sample_index']
        img_path = os.path.join(self.data_dir, fname)
        mask_path = img_path.replace("img_", "mask_")
        
        img = cv2.imread(img_path)
        if img is None: return None, None
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = (img/255.0).astype(np.float32)
        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None: mask = np.zeros(img.shape[:2], dtype=np.uint8)
        _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
        
        return img, mask

# --- 5. DATA PREPARATION (PATIENT-AWARE SPLIT) ---
print("📊  Configuring Patient-Aware Stratified Split...")
df_full = pd.read_csv(TRAIN_CSV_PATH)

# Strategy: Extract Patient ID from filename to prevent data leakage.
# Patches from the same patient must remain in the same split (Train OR Val).
df_full['patient_id'] = df_full['sample_index'].apply(lambda x: '_'.join(x.split('_')[:2]))

unique_patients = df_full.drop_duplicates(subset='patient_id')[['patient_id', 'label']]
train_patients, val_patients = train_test_split(
    unique_patients['patient_id'], 
    test_size=0.2, 
    stratify=unique_patients['label'], 
    random_state=42
)

train_df = df_full[df_full['patient_id'].isin(train_patients)].reset_index(drop=True)
val_df = df_full[df_full['patient_id'].isin(val_patients)].reset_index(drop=True)

# Training Augmentation Pipeline
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1)
])

train_loader = DataLoader(GrumpyDataset('train', train_df, TRAIN_CROPS_DIR, train_tf), 
                          batch_size=32, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(GrumpyDataset('train', val_df, TRAIN_CROPS_DIR, None), 
                        batch_size=32, shuffle=False, num_workers=2)

print(f"   Train Set: {len(train_df)} patches | Val Set: {len(val_df)} patches")

# --- 6. ARCHITECTURE SETUP (ResNet50 + Custom Head) ---
model = resnet50(weights=ResNet50_Weights.DEFAULT)

# Architecture Modification: 
# Replace default FC layer with a custom block including a 512-unit hidden layer
model.fc = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.fc.in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(512, 4)
)
model = model.to(device)

# Optimization Configuration
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-6)
scaler = torch.cuda.amp.GradScaler() # Mixed precision

# Loss Function with Class Weights and Label Smoothing
weights = torch.tensor([1.65, 1.2, 1.7, 4.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

# --- 7. TRAINING LOOP ---
EPOCHS = 100
PATIENCE = 30
best_val_f1 = -1.0
patience_counter = 0

print(f"\n🔥  Starting Training ({EPOCHS} epochs)...")

for epoch in range(1, EPOCHS+1):
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Ep {epoch}", leave=False)
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        
        # Regularization: MixUp Augmentation
        use_mixup = False
        if np.random.rand() < 0.5:
            use_mixup = True
            lam = np.random.beta(0.2, 0.2)
            idx = torch.randperm(inputs.size(0)).to(device)
            mixed_input = lam * inputs + (1 - lam) * inputs[idx]
            t_a, t_b = targets, targets[idx]
        
        with torch.cuda.amp.autocast():
            if use_mixup:
                out = model(mixed_input)
                loss = lam * criterion(out, t_a) + (1 - lam) * criterion(out, t_b)
            else:
                out = model(inputs)
                loss = criterion(out, targets)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        
    # Validation Phase
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            with torch.cuda.amp.autocast():
                out = model(inputs)
            preds.extend(out.argmax(1).cpu().numpy())
            trues.extend(targets.cpu().numpy())
            
    val_f1 = f1_score(trues, preds, average='micro')
    print(f"   Ep {epoch} | Loss: {running_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f}")
    
    # Early Stopping & Checkpointing
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), "best_model_v104.pth")
        print("   💾 Best Model Saved")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("🛑 Early Stopping Triggered")
            break

# --- 8. INFERENCE (Mask-Guided Tiling on RAW Test Images) ---
print("\n🔮  Running Inference on Test Set...")
if os.path.exists("best_model_v104.pth"):
    model.load_state_dict(torch.load("best_model_v104.pth"))
model.eval()

# Helper tensors for normalization
norm_mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
norm_std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
inv_map = {0: 'Luminal A', 1: 'Luminal B', 2: 'HER2(+)', 3: 'Triple negative'}

# Load test file list
test_files = sorted([f for f in os.listdir(TEST_RAW_DIR) if f.startswith('img_') and f.endswith('.png')])
df_test = pd.DataFrame({'sample_index': test_files})
ds_test = GrumpyDataset('test', df_test, TEST_RAW_DIR)

results = []

for idx in tqdm(range(len(ds_test))):
    fname = ds_test.df.iloc[idx]['sample_index']
    
    # 1. Load Raw Image & Mask
    full_img, full_mask = ds_test.get_raw_test_data(idx)
    
    # Handle edge case where image load fails
    if full_img is None: 
        results.append({'sample_index': fname, 'label': 'Luminal B'})
        continue
        
    H, W, _ = full_img.shape
    
    # 2. Dynamic Tile Generation
    # Strategy: Sliding window (224x224) with stride 112.
    # Filtering: Only keep patches where tissue mask coverage > 10%.
    patches = []
    
    for y in range(0, H - 224 + 1, 112):
        for x in range(0, W - 224 + 1, 112):
            mask_patch = full_mask[y:y+224, x:x+224]
            # Threshold check
            if np.sum(mask_patch > 0) > (224 * 224 * 0.1): # 10% valid tissue
                p = full_img[y:y+224, x:x+224]
                # Preprocessing
                p_t = torch.from_numpy(p).permute(2,0,1)
                p_t = (p_t - norm_mean) / norm_std
                patches.append(p_t)
    
    # 3. Patch Aggregation & Prediction
    pred_label = "Luminal B" # Default fallback
    
    if len(patches) > 0:
        batch = torch.stack(patches).to(device)
        
        # Batch inference for efficiency
        probs_list = []
        with torch.no_grad():
            for i in range(0, len(batch), 32):
                logits = model(batch[i:i+32])
                probs_list.append(F.softmax(logits, dim=1))
        
        all_probs = torch.cat(probs_list, dim=0)
        
        # Aggregation: Top-K Soft Voting (Top 50%)
        # Only considers the most confident patches to reduce background noise influence
        k = max(1, int(len(all_probs) * 0.5))
        topk_vals, _ = torch.topk(all_probs, k, dim=0)
        avg_probs = topk_vals.mean(dim=0).cpu().numpy()
        
        pred_label = inv_map[np.argmax(avg_probs)]
        
    results.append({'sample_index': fname, 'label': pred_label})

# Export Results
pd.DataFrame(results).to_csv("submission.csv", index=False)
print("✅  Inference Complete. submission.csv generated.")

⚙️  Hardware: Tesla P100-PCIE-16GB

⬇️  Downloading Teammate's Data...


Downloading...
From (original): https://drive.google.com/uc?id=1Kit41dsZPNYHNJHml6Wp7JK1yHom1VwK
From (redirected): https://drive.google.com/uc?id=1Kit41dsZPNYHNJHml6Wp7JK1yHom1VwK&confirm=t&uuid=d7d963db-3bae-4f88-9407-ffad21582f07
To: /kaggle/working/grumpy-data/crops_data.zip
100%|██████████| 274M/274M [00:02<00:00, 109MB/s]
Downloading...
From: https://drive.google.com/uc?id=17XPlHzQsI4_CUFJletOlVvWRdMuAFkzS
To: /kaggle/working/grumpy-data/crops_labels.csv
100%|██████████| 96.7k/96.7k [00:00<00:00, 59.0MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1UTRi8b5z_MQSis0IS6L91BqFisTVRgHj
From (redirected): https://drive.google.com/uc?id=1UTRi8b5z_MQSis0IS6L91BqFisTVRgHj&confirm=t&uuid=8065405c-bd15-4de9-a38c-c9cf5087869e
To: /kaggle/working/grumpy-data/test_data.zip
100%|██████████| 420M/420M [00:01<00:00, 309MB/s]


📦  Extracting...

📂  Training on Crops: /kaggle/working/grumpy-data/train_data_crops_crop224_ov56
📂  Testing on Raw:    /kaggle/working/grumpy-data/test_data
📊  Preparing Patient-Aware Split...
   Train Crops: 2872 | Val Crops: 690


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 230MB/s]



🔥  Training v1.0.4 for 100 epochs...


...

   Ep 57 | Loss: 0.9172 | Val F1: 0.3232


Ep 58:   0%|          | 0/89 [00:00<?, ?it/s]

   Ep 58 | Loss: 0.8800 | Val F1: 0.3261
🛑 Early Stopping

🔮  Inference...


  0%|          | 0/477 [00:00<?, ?it/s]

✅  Done! submission.csv created.
